# Daily Brief Builder

A single notebook that pulls today's Notion tasks + workout, latest webcomic strips, recent music releases, and news on followed topics, then renders a styled HTML "newspaper" to `output/daily_brief.html`.

Run locally for a preview, or let GitHub Actions run it daily and commit the output back to the repo (Pages serves it).


## 1 · Configuration

Everything you'd change over time lives here. The notebook reads nothing else for preferences.


In [ ]:
CONFIG = {
    "owner_name": "Mayank",
    "masthead_title": "THE DAILY BRIEF",
    "timezone": "Asia/Kolkata",

    # Which workout loadout to pull right now.
    # Must match an option on the Notion "Loadout" multi-select exactly.
    "loadout": "1 · Full Gym",

    # Webcomic RSS feeds. Shows the latest N entries per feed from the last `comic_lookback_days`.
    "webcomic_feeds": [
        {"name": "xkcd", "url": "https://xkcd.com/rss.xml"},
        {"name": "SMBC", "url": "https://www.smbc-comics.com/comic/rss"},
        {"name": "Oglaf", "url": "https://www.oglaf.com/feeds/rss/"},
    ],
    "comic_lookback_days": 2,
    "comic_items_per_feed": 2,

    # Bands/artists to watch via MusicBrainz. Any release (album/EP/single)
    # from the last N days shows up.
    "bands": [
        "Metallica",
        "Iron Maiden",
        "Ghost",
    ],
    "music_lookback_days": 60,

    # News topics via Google News RSS.
    "news_topics": [
        "Witcher",
        "Harry Potter",
    ],
    "news_items_per_topic": 5,
    "news_lookback_days": 2,
    "news_hl": "en-IN",     # Google News hl / gl / ceid
    "news_gl": "IN",
    "news_ceid": "IN:en",
}


## 2 · Imports


In [ ]:
import os
import re
import json
import time
import html
from pathlib import Path
from datetime import datetime, timedelta, timezone

import requests
import feedparser
from dateutil import parser as dateparser
from jinja2 import Template


## 3 · Paths and secrets

Tokens come from environment variables when running in CI, or from `.txt` files in the project root when running locally.


In [ ]:
# Resolve project root whether the notebook is run from scripts/ or project root.
HERE = Path.cwd()
PROJECT_ROOT = HERE if (HERE / "scripts").exists() else HERE.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

def load_token(env_name: str, filename: str) -> str:
    if env_name in os.environ and os.environ[env_name].strip():
        return os.environ[env_name].strip()
    return (PROJECT_ROOT / filename).read_text().strip()

NOTION_TASKS_TOKEN = load_token("NOTION_TASKS_TOKEN", "notion_tasks_token.txt")
NOTION_WORKOUT_TOKEN = load_token("NOTION_WORKOUT_TOKEN", "notion_workoutdash_token.txt")

TASKS_DATA_SOURCE_ID = "2e2bec68-aefa-81be-bfda-000b7366ef2d"
WORKOUT_DATA_SOURCE_ID = "28676ddd-d907-4643-aa88-10ffd7e38889"

NOTION_VERSION = "2026-03-11"


## 4 · Date helpers


In [ ]:
# All date logic uses the configured timezone.
from zoneinfo import ZoneInfo
TZ = ZoneInfo(CONFIG["timezone"])
NOW = datetime.now(TZ)
TODAY = NOW.date()
WEEKDAY_NAME = NOW.strftime("%A")  # e.g. "Saturday"

def ordinal(n: int) -> str:
    if 11 <= n % 100 <= 13:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"

DATE_LONG = f"{NOW.strftime('%A')}, {NOW.strftime('%B')} {ordinal(NOW.day)}, {NOW.year}"
DATE_SHORT = NOW.strftime("%Y-%m-%d")
VOLUME = NOW.year - 2025
ISSUE = NOW.timetuple().tm_yday

print(f"Building brief for {DATE_LONG} (weekday: {WEEKDAY_NAME}, loadout: {CONFIG['loadout']})")


## 5 · Notion client

Thin wrapper around `POST /v1/data_sources/{id}/query`. Pagination handled.


In [ ]:
NOTION_API = "https://api.notion.com/v1"

def notion_query(data_source_id: str, token: str, filter_body: dict | None = None) -> list[dict]:
    headers = {
        "Authorization": f"Bearer {token}",
        "Notion-Version": NOTION_VERSION,
        "Content-Type": "application/json",
    }
    url = f"{NOTION_API}/data_sources/{data_source_id}/query"
    results = []
    payload: dict = {}
    if filter_body:
        payload["filter"] = filter_body
    while True:
        r = requests.post(url, headers=headers, json=payload, timeout=30)
        r.raise_for_status()
        data = r.json()
        results.extend(data["results"])
        if not data.get("has_more"):
            break
        payload["start_cursor"] = data["next_cursor"]
    return results

def plain_text(rich: list[dict]) -> str:
    return "".join(part.get("plain_text", "") for part in (rich or []))


## 6 · Today's tasks

Pulls everything with `Task Date == TODAY`.


In [ ]:
def fetch_tasks() -> list[dict]:
    rows = notion_query(
        TASKS_DATA_SOURCE_ID,
        NOTION_TASKS_TOKEN,
        filter_body={"property": "Task Date", "date": {"equals": DATE_SHORT}},
    )
    tasks = []
    for row in rows:
        props = row["properties"]
        title = plain_text(props["Task"]["title"])
        done = props["Done"]["checkbox"]
        type_val = (props["Type"]["select"] or {}).get("name")
        tasks.append({
            "title": title or "(untitled)",
            "done": done,
            "type": type_val,
            "type_class": re.sub(r"[^A-Za-z]", "", type_val or ""),
        })
    # Finished at the bottom, priorities at the top.
    priority_order = {"PRIORITY": 0, "WORK": 1, "PERSONAL": 2, "RANGE": 3, "RECURRENT": 4, None: 5}
    tasks.sort(key=lambda t: (t["done"], priority_order.get(t["type"], 9), t["title"].lower()))
    return tasks

tasks = fetch_tasks()
print(f"Found {len(tasks)} task(s) for {DATE_SHORT}")
for t in tasks:
    print(f"  [{t['type'] or '-'}] {t['title']} {'(done)' if t['done'] else ''}")


## 7 · Today's workout

Pulls rows where `Day` contains today's weekday, `Loadout` contains the configured loadout, and `In Rotation` is checked.


In [ ]:
def fetch_workouts() -> list[dict]:
    rows = notion_query(
        WORKOUT_DATA_SOURCE_ID,
        NOTION_WORKOUT_TOKEN,
        filter_body={"and": [
            {"property": "Day", "multi_select": {"contains": WEEKDAY_NAME}},
            {"property": "Loadout", "multi_select": {"contains": CONFIG["loadout"]}},
            {"property": "In Rotation", "checkbox": {"equals": True}},
        ]},
    )
    workouts = []
    for row in rows:
        p = row["properties"]
        workouts.append({
            "name": plain_text(p["Name"]["title"]) or "(untitled)",
            "type": (p["Type"]["select"] or {}).get("name"),
            "sets": p["Sets"]["number"],
            "reps": plain_text(p["Reps"]["rich_text"]),
            "weight_kg": p["Current Weight (kg)"]["number"],
            "weight_lb": (p["Current Weight (lb)"]["formula"] or {}).get("number"),
            "rest": p["Rest (sec)"]["number"],
            "muscles": [opt["name"] for opt in p["Muscle Groups"]["multi_select"]],
            "progression": plain_text(p["Progression"]["rich_text"]),
        })
    # Strength first, then Core, Cardio, Mobility; stable within groups.
    type_order = {"Strength": 0, "Core": 1, "Cardio": 2, "Mobility": 3, None: 4}
    workouts.sort(key=lambda w: type_order.get(w["type"], 9))
    return workouts

workouts = fetch_workouts()
print(f"Found {len(workouts)} workout(s) for {WEEKDAY_NAME} on loadout {CONFIG['loadout']!r}")
for w in workouts:
    print(f"  {w['name']} — {w['sets']}x{w['reps']} @ {w['weight_kg']}kg")


## 8 · Webcomic RSS

Takes up to `comic_items_per_feed` of the most recent strips from each configured feed, keeping only those published within `comic_lookback_days` days.


In [ ]:
def entry_dt(entry):
    for attr in ("published", "updated", "created"):
        v = entry.get(attr)
        if v:
            return dateparser.parse(v).astimezone(TZ)
    return None

def fetch_comics() -> list[dict]:
    cutoff = NOW - timedelta(days=CONFIG["comic_lookback_days"])
    out = []
    for feed_cfg in CONFIG["webcomic_feeds"]:
        parsed = feedparser.parse(feed_cfg["url"])
        kept = 0
        for entry in parsed.entries:
            dt = entry_dt(entry)
            if dt is None or dt < cutoff:
                continue
            out.append({
                "source": feed_cfg["name"],
                "title": entry.get("title", "Untitled strip"),
                "url": entry.get("link", "#"),
                "when": dt.strftime("%b %-d, %H:%M"),
                "_dt": dt,
            })
            kept += 1
            if kept >= CONFIG["comic_items_per_feed"]:
                break
    out.sort(key=lambda c: c["_dt"], reverse=True)
    for c in out:
        c.pop("_dt", None)
    return out

comics = fetch_comics()
print(f"Found {len(comics)} comic strip(s) within last {CONFIG['comic_lookback_days']} day(s)")
for c in comics:
    print(f"  [{c['source']}] {c['title']} — {c['when']}")


## 9 · Music releases via MusicBrainz

Free, no auth. For each band in the config, finds the MBID with a text search, then lists release-groups with a first-release-date in the lookback window.

MusicBrainz asks for a descriptive User-Agent and ~1 request/sec rate.


In [ ]:
MB_API = "https://musicbrainz.org/ws/2"
MB_HEADERS = {"User-Agent": "DailyBrief/1.0 (personal use)"}

def mb_get(path: str, params: dict) -> dict:
    params = {**params, "fmt": "json"}
    r = requests.get(f"{MB_API}/{path}", params=params, headers=MB_HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def find_artist_mbid(name: str) -> str | None:
    data = mb_get("artist", {"query": f'artist:"{name}"', "limit": 1})
    artists = data.get("artists", [])
    if not artists:
        return None
    return artists[0]["id"]

def fetch_music() -> list[dict]:
    cutoff = (NOW - timedelta(days=CONFIG["music_lookback_days"])).date()
    out = []
    for band in CONFIG["bands"]:
        mbid = find_artist_mbid(band)
        time.sleep(1.1)
        if not mbid:
            continue
        data = mb_get("release-group", {
            "artist": mbid,
            "limit": 25,
            "offset": 0,
        })
        time.sleep(1.1)
        for rg in data.get("release-groups", []):
            date_str = rg.get("first-release-date") or ""
            if not date_str:
                continue
            try:
                rel_date = dateparser.parse(date_str).date()
            except (ValueError, dateparser.ParserError):
                continue
            if rel_date < cutoff:
                continue
            out.append({
                "artist": band,
                "title": rg.get("title", "Untitled"),
                "type": rg.get("primary-type") or "Release",
                "date": rel_date.strftime("%b %-d, %Y"),
                "url": f"https://musicbrainz.org/release-group/{rg['id']}",
                "_dt": rel_date,
            })
    out.sort(key=lambda m: m["_dt"], reverse=True)
    for m in out:
        m.pop("_dt", None)
    return out

music = fetch_music()
print(f"Found {len(music)} release(s) in the last {CONFIG['music_lookback_days']} day(s)")
for m in music:
    print(f"  [{m['artist']}] {m['title']} ({m['type']}, {m['date']})")


## 10 · News via Google News RSS

One section per topic, up to `news_items_per_topic` items each, filtered to the lookback window.


In [ ]:
from urllib.parse import quote_plus

def fetch_news() -> list[dict]:
    cutoff = NOW - timedelta(days=CONFIG["news_lookback_days"])
    sections = []
    for topic in CONFIG["news_topics"]:
        url = (
            f"https://news.google.com/rss/search?q={quote_plus(topic)}"
            f"&hl={CONFIG['news_hl']}&gl={CONFIG['news_gl']}&ceid={CONFIG['news_ceid']}"
        )
        parsed = feedparser.parse(url)
        items = []
        for entry in parsed.entries:
            dt = entry_dt(entry)
            if dt is None or dt < cutoff:
                continue
            title = entry.get("title", "Untitled")
            # Google News titles end with " - Source"; split cleanly.
            source = ""
            if " - " in title:
                title, source = title.rsplit(" - ", 1)
            items.append({
                "title": title,
                "url": entry.get("link", "#"),
                "source": source or "Google News",
                "when": dt.strftime("%b %-d, %H:%M"),
            })
            if len(items) >= CONFIG["news_items_per_topic"]:
                break
        sections.append({"name": topic, "entries": items})
    return sections

news = fetch_news()
for sec in news:
    print(f"[{sec['name']}] {len(sec['entries'])} item(s)")
    for n in sec["entries"]:
        print(f"  {n['title']} — {n['source']}")


## 11 · Render HTML

Reads the Jinja template and CSS from `scripts/`, inlines them, writes `output/daily_brief.html`.


In [ ]:
template_html = (SCRIPTS_DIR / "template_html.html").read_text()
template_css = (SCRIPTS_DIR / "template_css.css").read_text()

template = Template(template_html)
rendered = template.render(
    masthead_title=CONFIG["masthead_title"],
    owner_name=CONFIG["owner_name"],
    date_long=DATE_LONG,
    date_short=DATE_SHORT,
    volume=VOLUME,
    issue=ISSUE,
    weekday=WEEKDAY_NAME,
    loadout=CONFIG["loadout"],
    built_at=NOW.strftime("%Y-%m-%d %H:%M %Z"),
    css=template_css,
    tasks=tasks,
    workouts=workouts,
    comics=comics,
    music=music,
    news=news,
)

out_path = OUTPUT_DIR / "daily_brief.html"
out_path.write_text(rendered)
print(f"Wrote {out_path} ({len(rendered):,} bytes)")
